In [ ]:
import os
import copy
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import IsotonicRegression
from sklearn.metrics import roc_auc_score, log_loss
from tqdm.auto import tqdm
import lightgbm as lgb

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

class Config:
    BASE_DIR = r"C:\Users\narak\Downloads\sup_ai\ctr-ad-predict"
    train_path = os.path.join(BASE_DIR, "train.csv")
    test_path = os.path.join(BASE_DIR, "test.csv")
    sample_sub_path = os.path.join(BASE_DIR, "sample_submission.csv")
    output_path = os.path.join(BASE_DIR, "outputs", "submission_V10_Pinnacle.csv")

    target = 'clicked'
    num_cols = ['ad_quality_score', 'user_depth', 'historical_user_ctr']
    cat_cols = ['hour', 'day_of_week', 'device_type', 'device_model', 'device_conn_type',
                'banner_pos', 'is_app', 'creative_size', 'user_segment']
    
    batch_size = 8192 
    epochs = 40 
    lr = 1e-3
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_workers = 0 
    
    clip_min = 1e-6
    clip_max = 1.0 - 1e-6
    ema_alpha = 0.3 

In [ ]:
class PinnaclePreprocessor:
    def __init__(self, rare_threshold=5):
        self.rare_threshold = rare_threshold
        self.vocab_sizes = {}
        self.embed_dims = {}
        self.scaler = StandardScaler()
        self.final_num_cols = Config.num_cols.copy()
        self.final_cat_cols = Config.cat_cols.copy()

    def fit_transform(self, df_train, df_test):
        print("---  Starting Pinnacle Preprocessor V10 ---")
        
        # 1. Strict Time Sorting
        df_train['timestamp'] = pd.to_numeric(df_train['timestamp'], errors='coerce')
        df_test['timestamp'] = pd.to_numeric(df_test['timestamp'], errors='coerce')
        df_train = df_train.sort_values('timestamp').reset_index(drop=True)
        df_test = df_test.sort_values('timestamp').reset_index(drop=True)
            
        df_train['is_test'], df_test['is_test'], df_test[Config.target] = 0, 1, 0
        df_all = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
        
        # 2. Platform Logic & Crosses
        is_app_mask = df_all['is_app'].astype(str).str.lower().isin(['1', 'true', 'yes'])
        df_all['platform_id'] = np.where(is_app_mask, df_all['app_id'], df_all['site_id'])
        df_all['platform_domain'] = np.where(is_app_mask, df_all['app_domain'], df_all['site_domain'])
        
        df_all['ad_hour'] = df_all['ad_id'].astype(str) + "_" + df_all['hour'].astype(str)
        df_all['user_device'] = df_all['user_id'].astype(str) + "_" + df_all['device_type'].astype(str)
        
        for col in ['platform_id', 'platform_domain', 'ad_hour', 'user_device']:
            if col not in self.final_cat_cols: self.final_cat_cols.append(col)

        print("2/4 Calculating EMA Target Encoding (Time Decay)...")
        df_train_tmp = df_all[df_all['is_test'] == 0].copy()
        df_test_tmp = df_all[df_all['is_test'] == 1].copy()
        global_mean = df_train_tmp[Config.target].mean()
        
        ctr_targets = [
            ("user_id", "user_ctr"), ("ad_id", "ad_ctr"), ("platform_id", "platform_ctr"),
            (["user_id", "ad_id"], "user_ad_ctr"), (["ad_id", "platform_id"], "ad_platform_ctr")
        ]
        
        #  EMA TARGET ENCODING (Leak-Free & Time-Aware)
        for cols, new_col in ctr_targets:
            ema_series = df_train_tmp.groupby(cols)[Config.target].transform(
                lambda x: x.shift(1).ewm(alpha=Config.ema_alpha, min_periods=1).mean()
            )
            df_train_tmp[new_col] = ema_series.fillna(global_mean)
            
            last_known_ema = df_train_tmp.groupby(cols)[new_col].last().reset_index()
            df_test_tmp = df_test_tmp.merge(last_known_ema, on=cols, how='left')
            df_test_tmp[new_col] = df_test_tmp[new_col].fillna(global_mean)
            
            if new_col not in self.final_num_cols: self.final_num_cols.append(new_col)
            
        df_all = pd.concat([df_train_tmp, df_test_tmp], axis=0).reset_index(drop=True)

        print("3/4 Dictionary Mapping...")
        self.final_cat_cols = [c for c in self.final_cat_cols if c in df_all.columns]
        for col in self.final_cat_cols:
            df_all[col] = df_all[col].fillna('MISSING').astype(str)
            counts = df_all[col].value_counts()
            rare_classes = set(counts[counts < self.rare_threshold].index)
            df_all[col] = df_all[col].apply(lambda x: 'OOV' if x in rare_classes else x)
            
            val_to_idx = {val: i for i, val in enumerate(df_all[col].unique())}
            df_all[col] = df_all[col].map(val_to_idx)
            self.vocab_sizes[col] = len(val_to_idx)
            self.embed_dims[col] = min(50, max(4, int((self.vocab_sizes[col] ** 0.25) * 4)))
            
        print("4/4 Normalizing & Cleanup...")
        self.final_num_cols = [c for c in self.final_num_cols if c in df_all.columns]
        df_all[self.final_num_cols] = self.scaler.fit_transform(df_all[self.final_num_cols].fillna(0))
        
        drop_cols = ['is_test', 'impression_id', 'timestamp', 'user_id', 'ad_id', 'site_id', 'app_id', 
                     'site_domain', 'app_domain', 'C1', 'C15', 'C16', 'C21']
        df_all = df_all.drop(columns=[c for c in drop_cols if c in df_all.columns])
        
        print(" Preprocessing V10 Completed!")
        return df_all.iloc[:len(df_train_tmp)], df_all.iloc[len(df_train_tmp):]

In [ ]:
class CTRDataset(Dataset):
    def __init__(self, df, feature_names, target_col):
        self.x, self.y = df[feature_names].values.astype(np.float32), df[target_col].values.astype(np.float32)
    def __len__(self): return len(self.x)
    def __getitem__(self, idx): return self.x[idx], self.y[idx]

def build_dataloader(df, feature_names, target_col, batch_size, is_train=True):
    return DataLoader(CTRDataset(df, feature_names, target_col), batch_size=batch_size, shuffle=is_train, num_workers=Config.num_workers, pin_memory=True)

class PureFCN(nn.Module):
    def __init__(self, cat_cols, num_cols, vocab_sizes, embed_dims, hidden_units=(256, 128, 64), dropout=0.5):
        super(PureFCN, self).__init__()
        self.cat_cols, self.num_cols, self.num_cat = cat_cols, num_cols, len(cat_cols)
        self.embeddings = nn.ModuleDict({col: nn.Embedding(num_embeddings=vocab_sizes[col], embedding_dim=embed_dims[col]) for col in cat_cols})
        
        input_dim = sum(embed_dims[col] for col in cat_cols) + len(num_cols)
        layers = []
        for hidden_dim in hidden_units:
            layers.extend([nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
            input_dim = hidden_dim 
            
        self.mlp = nn.Sequential(*layers)
        self.output_layer = nn.Linear(input_dim, 1)

    def forward(self, x):
        cat_embs = [self.embeddings[col](x[:, i].long()) for i, col in enumerate(self.cat_cols)]
        cat_out = torch.cat(cat_embs, dim=1) if cat_embs else torch.empty(x.size(0), 0, device=x.device)
        return self.output_layer(self.mlp(torch.cat([cat_out, x[:, self.num_cat:]], dim=1)))

In [ ]:
def train_pure_fcn(model, train_loader, val_loader, epochs, lr, pos_weight):
    model.to(Config.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32).to(Config.device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    
    best_auc, best_weights, patience, epochs_no_improve = 0.0, None, 8, 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for batch_x, batch_y in train_pbar:
            batch_x, batch_y = batch_x.to(Config.device), batch_y.to(Config.device)
            optimizer.zero_grad()
            if torch.cuda.is_available():
                with torch.amp.autocast('cuda'):
                    loss = criterion(model(batch_x), batch_y.unsqueeze(1).float())
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = criterion(model(batch_x), batch_y.unsqueeze(1).float())
                loss.backward()
                optimizer.step()
            train_loss += loss.item()
            train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                probs = np.clip(torch.sigmoid(model(batch_x.to(Config.device))).cpu().numpy().flatten(), Config.clip_min, Config.clip_max)
                val_preds.extend(probs) 
                val_targets.extend(batch_y.numpy().flatten())
                
        val_auc, val_logloss = roc_auc_score(val_targets, val_preds), log_loss(val_targets, val_preds)
        print(f"Epoch {epoch+1}/{epochs} | LR: {optimizer.param_groups[0]['lr']:.1e} | Loss: {train_loss/len(train_loader):.4f} | Val AUC: {val_auc:.4f} | Val LogLoss: {val_logloss:.4f}")
        
        scheduler.step(val_auc)
        if val_auc > best_auc:
            best_auc, best_weights, epochs_no_improve = val_auc, copy.deepcopy(model.state_dict()), 0
            print("  [⭐] Best model updated!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  [!] Early stopping.")
                break

    model.load_state_dict(best_weights)
    return model

In [ ]:
print(f' Using Device: {Config.device}')

train_df = pd.read_csv(Config.train_path)
test_df = pd.read_csv(Config.test_path)
sub_df = pd.read_csv(Config.sample_sub_path)

preprocessor = PinnaclePreprocessor()
train_proc, test_proc = preprocessor.fit_transform(train_df, test_df)

#  3-WAY SPLIT: Train (75%) | Valid (15%) | Holdout (10%)
split_idx_1 = int(len(train_proc) * 0.75)
split_idx_2 = int(len(train_proc) * 0.90)

train_set = train_proc.iloc[:split_idx_1].copy()
val_set   = train_proc.iloc[split_idx_1:split_idx_2].copy()
hold_set  = train_proc.iloc[split_idx_2:].copy() 
    
print(f"Train size: {len(train_set):,}, Val size: {len(val_set):,}, Holdout size: {len(hold_set):,}")

num_pos = train_set[Config.target].sum()
num_neg = len(train_set) - num_pos
pos_weight_val = np.sqrt(num_neg / num_pos) 

feature_names = preprocessor.final_cat_cols + preprocessor.final_num_cols

train_loader = build_dataloader(train_set, feature_names, Config.target, batch_size=Config.batch_size, is_train=True)
val_loader = build_dataloader(val_set, feature_names, Config.target, batch_size=Config.batch_size, is_train=False)
hold_loader = build_dataloader(hold_set, feature_names, Config.target, batch_size=Config.batch_size, is_train=False)
test_loader = build_dataloader(test_proc, feature_names, Config.target, batch_size=Config.batch_size, is_train=False)

In [ ]:
if __name__ == '__main__':
    # --- [1] Pure FCN ---
    print("\n" + "="*40 + "\n MODEL 1: Training Pure FCN\n" + "="*40)
    fcn_model = PureFCN(preprocessor.final_cat_cols, preprocessor.final_num_cols, preprocessor.vocab_sizes, preprocessor.embed_dims)
    best_fcn = train_pure_fcn(fcn_model, train_loader, val_loader, epochs=Config.epochs, lr=Config.lr, pos_weight=pos_weight_val)
    
    best_fcn.eval()
    fcn_hold_preds, fcn_test_preds = [], []
    with torch.no_grad():
        for batch_x, _ in hold_loader: 
            fcn_hold_preds.extend(np.clip(torch.sigmoid(best_fcn(batch_x.to(Config.device))).cpu().numpy().flatten(), Config.clip_min, Config.clip_max))
        for batch_x, _ in test_loader: 
            fcn_test_preds.extend(np.clip(torch.sigmoid(best_fcn(batch_x.to(Config.device))).cpu().numpy().flatten(), Config.clip_min, Config.clip_max))
    fcn_hold_preds, fcn_test_preds = np.array(fcn_hold_preds), np.array(fcn_test_preds)

    # --- [2] LightGBM ---
    print("\n" + "="*40 + "\n MODEL 2: Training LightGBM\n" + "="*40)
    cat_idx = [feature_names.index(c) for c in preprocessor.final_cat_cols if c in feature_names]
    train_data = lgb.Dataset(train_set[feature_names], label=train_set[Config.target], categorical_feature=cat_idx)
    val_data = lgb.Dataset(val_set[feature_names], label=val_set[Config.target], reference=train_data, categorical_feature=cat_idx)
    
    lgb_params = {
        'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.01, 'num_leaves': 31, 'max_depth': 6,
        'min_data_in_leaf': 100, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
        'scale_pos_weight': pos_weight_val, 'lambda_l1': 0.1, 'lambda_l2': 2.0, 'verbose': -1, 'random_state': 42, 'n_jobs': -1
    }
    lgb_model = lgb.train(lgb_params, train_data, num_boost_round=3000, valid_sets=[train_data, val_data], 
                          callbacks=[lgb.early_stopping(100, verbose=True), lgb.log_evaluation(100)])
    
    lgb_hold_preds = np.clip(lgb_model.predict(hold_set[feature_names]), Config.clip_min, Config.clip_max)
    lgb_test_preds = np.clip(lgb_model.predict(test_proc[feature_names]), Config.clip_min, Config.clip_max)

    # --- [3] Blending & [4] Calibration (On HOLDOUT SET) ---
    print("\n" + "="*40 + "\n ENSEMBLE &  ISOTONIC CALIBRATION (Holdout Set)\n" + "="*40)
    y_hold_true = hold_set[Config.target].values
    best_weight, best_blend_auc = 0.5, 0.0
    
    for w in np.arange(0.0, 1.05, 0.05):
        blend_auc = roc_auc_score(y_hold_true, (w * lgb_hold_preds) + ((1 - w) * fcn_hold_preds))
        if blend_auc > best_blend_auc: best_blend_auc, best_weight = blend_auc, w
            
    print(f'[] Robust Optimal Blend -> LGBM: {best_weight:.2f} | FCN: {1 - best_weight:.2f} | HOLDOUT AUC: {best_blend_auc:.4f}')
    
    final_hold_preds = (best_weight * lgb_hold_preds) + ((1 - best_weight) * fcn_hold_preds)
    final_test_preds = (best_weight * lgb_test_preds) + ((1 - best_weight) * fcn_test_preds)

    iso_reg = IsotonicRegression(out_of_bounds='clip')
    iso_reg.fit(final_hold_preds, y_hold_true)
    
    calibrated_hold_preds = np.clip(iso_reg.predict(final_hold_preds), Config.clip_min, Config.clip_max)
    calibrated_test_preds = np.clip(iso_reg.predict(final_test_preds), Config.clip_min, Config.clip_max)
    
    sub_df[Config.target] = calibrated_test_preds
    os.makedirs(os.path.dirname(Config.output_path), exist_ok=True)
    sub_df.to_csv(Config.output_path, index=False)
    
    print(f"LogLoss AFTER Calibration & Clipping (Holdout): {log_loss(y_hold_true, calibrated_hold_preds):.5f}")
    print(f'\n V10 Pinnacle Sub saved to: {Config.output_path}')